### AGENTIC RAG WITH LANGGRAPH

In [2]:
import os
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

C:\Users\gargi\AppData\Local\Temp\ipykernel_24648\2682487706.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


ModuleNotFoundError: No module named 'langchain.text_splitter'

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

False

In [ ]:
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY")

llm = ChatOpenAI(model_name="gpt-4", temperature=0.7, max_tokens=1000)
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")   

In [ ]:
llm

### STATE DEFINITION

In [ ]:
class AgentState(TypedDict):
    query: str
    docs: List[Document]
    answer: str
    needs_retrieval: bool

In [ ]:
### sample documnetas and vectorstore

smaple_text = ["langgraph is a powerful tool for building and visualizing complex workflows. It allows developers to create state machines, decision trees, and other types of graphs that can be used to model complex processes. With langgraph, you can easily define states, transitions, and actions, and then visualize the resulting graph in a clear and intuitive way. This makes it easier to understand how your workflow operates and identify potential bottlenecks or areas for improvement.",
               "RAG (Retrieval-Augmented Generation) is a technique that combines the power of retrieval-based methods with generative models to improve the quality and relevance of generated responses. In RAG, a retrieval model is used to fetch relevant information from a large corpus of documents, which is then used to inform the generation process. This allows the model to generate more accurate and contextually appropriate responses, as it can leverage external knowledge sources to supplement its own understanding. RAG has been shown to be effective in a variety of applications, including question answering, dialogue systems, and text summarization."
               "Vector stores are specialized databases designed to efficiently store and retrieve high-dimensional vectors, which are often used in machine learning and data science applications. These vectors can represent various types of data, such as images, text, or audio, and are typically generated using techniques like embeddings or feature extraction. Vector stores enable fast similarity searches, allowing users to find the most similar items in a dataset based on their vector representations. They are commonly used in recommendation systems, image and video retrieval, natural language processing, and other applications that require efficient handling of large-scale vector data."
               "agents are autonomous software entities that can perform tasks or make decisions on behalf of users or other systems. They are designed to operate independently, often using artificial intelligence and machine learning techniques to adapt to changing environments and learn from experience. Agents can be found in various domains, including robotics, virtual assistants, and multi-agent systems, where they collaborate or compete to achieve specific goals. They can be programmed to perceive their environment, reason about it, and take actions to achieve their objectives."
]
documents = [Document(page_content=text) for text in smaple_text]

#### create a vectorstore
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2})

NameError: name 'Document' is not defined

### agnetic function

In [ ]:
def decide_retrival(state: AgentState) -> AgentState:
    """Decides if we need to retrive documents v=based on the query and the current state of the agent."""
    question = state["question"]
    #simple  heuristic : if question conatains certain keywords retrive
    retrieval_keywords = ["who", "what", "when", "where", "why", "how", "explain", "define"]
    needs_retrieval  = any(keyword in question.lower() for keyword in retrieval_keywords)
    state["needs_retrieval"] = needs_retrieval
    return state

In [ ]:
def retrieve_documents(state: AgentState) -> AgentState:
    """Retrieves documents based on the query and updates the state with the retrieved documents."""
    question = state["question"]
    documents = retriever.invoke(question)

    return {
        **state,
        "documents": documents
    }

In [ ]:
def generate_answer(state: AgentState) -> AgentState:
    """Generates an answer based on the query and the retrieved documents."""
    question = state["question"]
    documents = state.get("documents", [])
    
    # Combine the content of the retrieved documents into a single context string
    context = " ".join(doc.page_content for doc in documents)
    
    # Create a prompt for the LLM
    prompt = f"Question: {question}\nContext: {context}\nAnswer:"
    
    # Generate an answer using the LLM
    answer = llm.invoke(prompt)
    
    return {
        **state,
        "answer": answer
    }

### conditional logicc

In [ ]:
def should_retrieve(state: AgentState) -> bool:
    """Determines if document retrieval is necessary based on the current state."""
    if state["needs_retrieval"]:
        return "retrieve"
    else:
        return "generate"

### BUILD THE GRAPG

In [5]:
#create a state graph
workflow = StateGraph(AgentState)

#add noddws to the graph
workflow.add.node("decide", decide_retrival)
workflow.add.node("retrieve", retrieve_documents)
workflow.add.node("generate", generate_answer)

#set entryb point
workflow.set_entry_point("decide")

#add conditional edges
workflow.add_conditional_edges(
    "decide",
     should_retrieve, {
    "retrieve": "retrieve",
    "generate": "generate"
})

workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

app = workflow.compile()
app

NameError: name 'AgentState' is not defined

### TEST THE AGENTIC SYSTEM

In [ ]:
def ask_question(question: str) -> str:
    """Asks a question to the agent and returns the generated answer."""
    initial_state: AgentState = {
        "question": question,
        "documents": [],
        "answer": "",
        "needs_retrieval": False
    }
    
    final_state = app.invoke(initial_state)
    return final_state["answer"]